# 🖼️ Padding & Strides in Convolutional Neural Networks (CNN)

---

## 1. Executive Summary

In standard Convolutional operations, applying a kernel/filter directly to an input image introduces two major challenges:
1. **Shrinkage / Spatial Dimension Loss:** The feature map shrinks after every convolution layer, causing rapid information loss across deep architectures.
2. **Edge Information Loss:** Boundary/corner pixels are sampled significantly fewer times than central pixels, causing boundary-specific features to be underrepresented.

**Padding** and **Strides** serve as structural hyperparameters to control the output feature map's spatial dimensions, information retention, and computational complexity.

---

## 2. Issues with Standard Convolution (No Padding)

For an input image of size $n \times n$ and a filter/kernel of size $f \times f$, the resulting output feature map dimension without padding is:

$$\text{Output Size} = (n - f + 1) \times (n - f + 1)$$

### The Two Major Drawbacks
* **Dimension Shrinkage:** 
  * *Example:* A $5 \times 5$ image convolved with a $3 \times 3$ filter yields a $3 \times 3$ feature map.
  * Applying multiple sequential convolutions quickly reduces feature map dimensions to $1 \times 1$, limiting network depth.
* **Pixel Usage Bias (Loss of Corner/Edge Information):**
  * Central pixels overlap with the filter multiple times during sliding window passes.
  * Corner/edge pixels are covered only once, causing the network to under-weigh perimeter information.

---

## 3. Padding

Padding surrounds the input matrix with extra outer border layers (typically filled with $0$, known as **Zero Padding**) of thickness $p$.

$$\text{Padded Input Size} = (n + 2p) \times (n + 2p)$$

### Types of Padding
* **Valid Padding ($p = 0$):** No padding added. The feature map shrinks naturally.
  $$\text{Output Size} = (n - f + 1)$$
* **Same Padding:** Padding $p$ is configured so that the **output dimensions match the input dimensions** ($n \times n$).
  $$\text{Condition:} \quad n + 2p - f + 1 = n \implies p = \frac{f - 1}{2}$$
  *(Note: Filter sizes $f$ are typically chosen as odd numbers—such as $3 \times 3, 5 \times 5$—so that $p$ remains an integer).*

---

## 4. Strides

**Stride ($s$)** defines the step size by which the filter slides across the input image (horizontally and vertically).

* **Default Stride ($s = 1$):** Filter moves $1$ pixel at a time, resulting in dense feature maps.
* **Larger Strides ($s > 1$):** Filter skips pixels ($s$ steps per movement), downsampling spatial dimensions and reducing computational complexity.

---

## 5. Generalized Output Formula

Combining Input Dimension ($n$), Filter Size ($f$), Padding ($p$), and Strides ($s$), the exact output dimension of a convolutional layer is given by:

$$\text{Output Dimension} = \left\lfloor \frac{n + 2p - f}{s} \right\rfloor + 1$$

*(Note: The floor function $\lfloor \cdot \rfloor$ handles non-integer divisions by discarding extra fractional boundary steps where the filter does not completely fit).*

---

## 6. Comparative Overview

| Parameter | Primary Purpose | Formula Component | Key Effect |
| :--- | :--- | :--- | :--- |
| **Padding ($p$)** | Preserves spatial dimensions & edge information | $+ 2p$ inside numerator | Expands feature maps; prevents border pixel information loss |
| **Stride ($s$)** | Controls downsampling & computational speed | Divisor $s$ in denominator | Reduces feature map resolution; decreases computational load |

---

## 7. Implementation Reference (Keras / TensorFlow)

Padding and stride controls are built directly into standard Keras Conv2D layers:

```python
import tensorflow as tf
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.models import Sequential

model = Sequential([
    # Valid Padding (p=0), Stride s=1
    Conv2D(filters=32, kernel_size=(3, 3), strides=1, padding='valid', input_shape=(28, 28, 1)),
    
    # Same Padding (p calculated automatically), Stride s=1
    Conv2D(filters=64, kernel_size=(3, 3), strides=1, padding='same'),
    
    # Same Padding with Stride s=2 (Downsampling layer)
    Conv2D(filters=128, kernel_size=(3, 3), strides=2, padding='same')
])

model.summary()

In [2]:
import tensorflow 
from tensorflow import keras 
from keras.layers import Dense , Input , Conv2D, Flatten
from keras import Sequential 
from keras.datasets import mnist

In [4]:
(x_train, y_train),(y_train,y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


### Padding = Valid

In [6]:
model = Sequential([
    Input(shape=(28,28,1)),
    Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu'),
    Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu'),
    Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu') ,

    Flatten() ,

    Dense(128,activation='relu'),
    Dense(10,activation='softmax')
    
])

In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 26, 26, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 24, 24, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 22, 22, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 15488)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       1,982,592 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,002,698 (7.64 MB)

 Trainable params: 2,002,698 (7.64 MB)

 Non-trainable params: 0 (0.00 B)

### Padding = Same 

In [8]:
model = Sequential([
    Input(shape=(28,28,1)),
    Conv2D(32,kernel_size=(3,3),padding='same',activation='relu'),
    Conv2D(32,kernel_size=(3,3),padding='same',activation='relu'),
    Conv2D(32,kernel_size=(3,3),padding='same',activation='relu') ,

    Flatten() ,

    Dense(128,activation='relu'),
    Dense(10,activation='softmax')
    
])

In [9]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 28, 28, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 28, 28, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │       3,211,392 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,231,498 (12.33 MB)

 Trainable params: 3,231,498 (12.33 MB)

 Non-trainable params: 0 (0.00 B)

### Strides 

In [11]:
model = Sequential([
    Input(shape=(28,28,1)),
    Conv2D(32,kernel_size=(3,3),padding='same',strides=(2,2) ,activation='relu'),
    Conv2D(32,kernel_size=(3,3),padding='same',strides=(2,2) , activation='relu'),
    Conv2D(32,kernel_size=(3,3),padding='same',strides=(2,2) , activation='relu') ,

    Flatten() ,

    Dense(128,activation='relu'),
    Dense(10,activation='softmax')
    
])

In [12]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)                    │ (None, 14, 14, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 7, 7, 32)            │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_8 (Conv2D)                    │ (None, 4, 4, 32)            │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 128)                 │          65,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 85,770 (335.04 KB)

 Trainable params: 85,770 (335.04 KB)

 Non-trainable params: 0 (0.00 B)